In [ ]:
from ngsolve import *
from ngsolve.webgui import Draw
from netgen.occ import *
from netgen.geom2d import SplineGeometry
"""der Code simuliert die Kirchhoff-love Gl mit HHJ-methode und gibt am ende ein werte-gitter aus, bzw speichert dieses auch ab"""


l = 2 #m Länge
b = 1 #m Breite

radius = 0.025 # Radius der Kreise für Auflager
 
shape = MoveTo(0,0).Rectangle(l,b).Face()

shape.edges.name="free"

#circle = Circle((0,0),0.10).Face()
#circle.edges.name="circles"

auflager_liste = [(radius,radius),(l-radius,b-radius),(radius,b-radius),(l-radius,radius)]

# auflager_liste = [(-a+radius,-b+radius),(-a/2,b/2),(-a/2,-b/2),(a/2,-b/2)]
#auflager_liste = [(-0.3,-0.1),(-0.1,0)]

rect = shape 
for mittelpunkt in auflager_liste:
    circle = Circle(mittelpunkt,radius).Face()
    circle.edges.name = "dirichlet"
    rect = rect - circle

geo = OCCGeometry(rect,dim=2)

#Draw(rect)
mesh = Mesh(geo.GenerateMesh(maxh=l/40))

mesh.Curve(3)
#Draw(mesh)

In [6]:
E  = 70e9      #Glas ~ 70 GPa Elastizitätsmodul
nu = 0.22
t  = 0.003     #Dicke [m]
rho = 2500   #kg/m^3 Dichte einer Aluminiumplatte
g = 9.81

q = rho * t * g     #Eigengewicht der Platte - rechte Seite der PDE

Db = E*t**3/(12*(1-nu**2))

def D(A):
    return Db *((1-nu)*A+ nu*Trace(A)*Id(2))

def Dinv(A):
    return 1/Db * (1/(1-nu)*A - nu/(1-nu**2)*Trace(A)*Id(2))



In [7]:
order = 3

V = HDivDiv(mesh, order=order-1,dirichlet="dirichlet")
Q = H1(mesh, order=order, dirichlet="dirichlet")
X = V * Q

(sigma,w),(tau,v)= X.TnT()

n = specialcf.normal(2)

def tang(u):
    return u - (u*n)*n


#schwache Form von Kirchhoff-Love mit Hellan-Herrmann-Johnson-Method
#https://docu.ngsolve.org/ngs24/SaS/Kirchhoff_Love_plate.html

a = BilinearForm(X,symmetric=True)
a += InnerProduct(Dinv(sigma),tau) * dx
a += div(sigma)*Grad(v) * dx
a += div(tau) * Grad(w) * dx
a += - (sigma[n,:] * tang(Grad(v)) + tau[n,:] * tang(Grad(w))  ) * dx(element_boundary=True)
a.Assemble()

L = LinearForm(X)
L += q * v *dx
L.Assemble()

gf_solution = GridFunction(X)
gf_solution.vec.data = a.mat.Inverse(X.FreeDofs(),inverse="") * L.vec

gf_sigma, gf_w = gf_solution.components
#Draw(gf_sigma, mesh,name="sigma")
Draw(gf_w, mesh, name="disp",deformation=True, euler_angles=[-60,5,30])




WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {'camera': {'euler_angles': […

BaseWebGuiScene

In [8]:
#zum Weiterrechnen wollen wir auf die errechneten Daten zugreifen können

import numpy as np
import inspect

step = 40           #TODO warum ab 41 out of domain?

#erzeugt gitter für gewünschte werte
x_points = np.linspace(0,l, step)
y_points = np.linspace(0,b, step)
X, Y = np.meshgrid(x_points, y_points)


#speichert (x,y,z) in result_matrix für alle (x,y) in X,Y
result_matrix = np.zeros((step, step),dtype=object)
for i in range(step):
    for j in range(step):
        result_matrix[i, j] = (float(X[i,j]), float(Y[i,j]), gf_w(X[i, j], Y[i, j]))

#print(result_matrix)


#speichert die Werte in *.xyz-Datei
with open("ergebnisse.xyz", "w") as f:
    for i in range(step):
        for j in range(step):
            x, y, w = result_matrix[i, j]
            f.write(f"{x:.6f}\t{y:.6f}\t{w:.6e}\n")


